In [ ]:
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder

df = pd.read_csv("data/Road Safety Data - Accidents 2019.csv")
print("data before processing:", df.shape)
df.describe(include='all')
print("Head of data before processing: ")
df.head()
# Drop useless columns
cols_to_drop = [
    "Accident_Index", "Location_Easting_OSGR", "Location_Northing_OSGR",
    "Longitude", "Latitude", "LSOA_of_Accident_Location",
    "1st_Road_Number", "2nd_Road_Number"
]
df = df.drop(columns=cols_to_drop)

# Replace -1 with NaN and fill with "Unknown"
df = df.replace(-1, np.nan)
df = df.fillna("Unknown")

# Maps
severity_map = {1: "Fatal", 2: "Serious", 3: "Slight"}
road_type_map = {1: "Roundabout", 2: "OneWay", 3: "DualCarriageway", 6: "SingleCarriageway", 7: "SlipRoad", 9: "UnknownRoad"}
light_map = {1: "Daylight", 4: "Dark_Lit", 5: "Dark_Unlit", 6: "Dark_NoLighting", 7: "Dark_LitUnknown"}
weather_map = {1: "Fine", 2: "Raining", 3: "Snowing", 4: "Fog", 5: "HighWinds", 7: "UnknownWeather",
               8: "Raining+HighWinds", 9: "Snowing+HighWinds"}
surface_map = {1: "Dry", 2: "Wet", 3: "Snow", 4: "Frost", 5: "Flood", 6: "Oil", 7: "UnknownSurface"}
urban_map = {1: "Urban", 2: "Rural", 3: "Unallocated"}

df["Accident_Severity"] = df["Accident_Severity"].map(severity_map)
df["Road_Type"] = df["Road_Type"].map(road_type_map)
df["Light_Conditions"] = df["Light_Conditions"].map(light_map)
df["Weather_Conditions"] = df["Weather_Conditions"].map(weather_map)
df["Road_Surface_Conditions"] = df["Road_Surface_Conditions"].map(surface_map)
df["Urban_or_Rural_Area"] = df["Urban_or_Rural_Area"].map(urban_map)

df["Speed_limit"] = pd.to_numeric(df["Speed_limit"], errors="coerce")

# Discretize Speed
df["SpeedBin"] = pd.cut(
    df["Speed_limit"],
    bins=[0, 30, 50, 70, 120],
    labels=["Speed_0_30", "Speed_30_50", "Speed_50_70", "Speed_70_plus"]
)

# Time period extraction
def time_to_period(t):
    if t == "Unknown" or t is np.nan: return "Time_Unknown"
    hour = int(t.split(":")[0])
    if 5 <= hour < 12: return "Morning"
    if 12 <= hour < 17: return "Afternoon"
    if 17 <= hour < 21: return "Evening"
    return "Night"

df["TimePeriod"] = df["Time"].apply(time_to_period)

# Create transaction items
def create_items(row):
    return [
        row["Light_Conditions"] if not pd.isna(row["Light_Conditions"]) else "Light_Unknown",
        row["Weather_Conditions"] if not pd.isna(row["Weather_Conditions"]) else "Weather_Unknown",
        row["Road_Surface_Conditions"] if not pd.isna(row["Road_Surface_Conditions"]) else "Surface_Unknown",
        "Day_" + str(row["Day_of_Week"]) if not pd.isna(row["Day_of_Week"]) else "Day_Unknown",
        row["TimePeriod"] if not pd.isna(row["TimePeriod"]) else "Time_Unknown",
        row["Urban_or_Rural_Area"] if not pd.isna(row["Urban_or_Rural_Area"]) else "Area_Unknown",
        row["SpeedBin"] if not pd.isna(row["SpeedBin"]) else "Speed_Unknown",
        row["Accident_Severity"] if not pd.isna(row["Accident_Severity"]) else "Severity_Unknown",
        "RoadType_" + str(row["Road_Type"]) if not pd.isna(row["Road_Type"]) else "RoadType_Unknown"
    ]

df["Items"] = df.apply(create_items, axis=1)

# One-hot encoding for Apriori and FP-Growth
te = TransactionEncoder()
te_data = te.fit(df["Items"]).transform(df["Items"])
df_encoded = pd.DataFrame(te_data, columns=te.columns_)
df_encoded.to_csv("data/processed_accidents_2019.csv", index=False)

print("data after processing:", df.shape)
df_encoded.describe(include='all')
print("Head of data after processing: ")
df_encoded.head()